# 1、LLMChain的使用

举例1：

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
修复后的数学问题求解示例

问题分析：
1. 移除了未使用的modulefinder导入
2. LLMChain已被弃用，改用推荐的LCEL语法（基于Runnable）
3. 简化了代码结构，使用更现代的LangChain API
"""

from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

# 加载环境变量
dotenv.load_dotenv()

# 设置OpenAI API密钥和基础URL
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 1、创建大模型实例
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="你是一个数学高手，帮我解决如下的数学问题：{question}"
)

# 使用LCEL语法（推荐方式）替代过时的LLMChain
chain = prompt_template | chat_model

# 执行链并获取响应
response = chain.invoke({"question": "1 + 2 * 3 = ?"})

# 打印响应
print("响应结果:")
print(response.content)  # 使用.content访问消息内容

# 如果需要返回与旧版本类似的字典格式
result_dict = {"text": response.content}
print("\n字典格式结果:")
print(result_dict)

响应结果:
根据数学的运算顺序（先乘除后加减），我们先计算乘法部分：

1 + 2 * 3 = 1 + 6 = 7

所以，1 + 2 * 3 的结果是 7。

字典格式结果:
{'text': '根据数学的运算顺序（先乘除后加减），我们先计算乘法部分：\n\n1 + 2 * 3 = 1 + 6 = 7\n\n所以，1 + 2 * 3 的结果是 7。'}


举例2：使用ChatPromptTemplate及参数verbose的演示

In [10]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 加载环境变量
dotenv.load_dotenv()

# 设置OpenAI环境变量
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个数学高手"),
    ("human", "帮我解决如下的数学问题：{question}")
])

# 使用LCEL语法构建链
chain = prompt_template | chat_model | StrOutputParser()

# 执行链
response = chain.invoke({"question": "1 + 2 * 3 = ?"})

print("问题: 1 + 2 * 3 = ?")
print("答案:", response)

问题: 1 + 2 * 3 = ?
答案: 按照运算顺序，先进行乘法再进行加法。因此，计算步骤为：

1. 计算 2 * 3 = 6
2. 然后计算 1 + 6 = 7

所以，1 + 2 * 3 = 7。


# 2、顺序链之SimpleSequentialChain的使用

举例1：


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

chainA_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位精通各领域知识的知名教授"),
        ("human", "请你尽可能详细的解释一下：{knowledge}"),
    ]
)

chat_model = ChatOpenAI(model="gpt-4o-mini")

chainA_chains = LLMChain(llm=chat_model,
                         prompt=chainA_template,
                         verbose=True
                         )



# chainA_chains.invoke({"knowledge":"什么是LangChain？"})

问题: 什么是LangChain？
答案:
LangChain是一个针对构建应用程序的框架，特别是在自然语言处理（NLP）和大型语言模型（LLM）的领域。它旨在通过提供一系列工具和接口，帮助开发者更高效地构建与语言模型相关的应用程序。LangChain的出现使得开发者能够更轻松地将多种语言模型、数据源和API整合在一起，从而实现更复杂的功能。

### LangChain的主要组成部分

1. **链（Chains）：**
   LangChain的核心概念是“链”，即将多个动作（例如调用模型、处理输入和输出等）以顺序方式组合在一起。这可以是简单的文本生成链，也可以是复杂的多步骤处理。

2. **提示（Prompting）：**
   在与语言模型交互时，提示是至关重要的。LangChain提供了用于构建和优化提示的工具，以确保传递给模型的内容能够达到预期效果。这包括动态生成提示，拼接或修改文本等功能。

3. **内存（Memory）：**
   LangChain支持状态管理，即内存的使用。能够记住用户的输入和上下文信息，对于需要保持对话状态的应用（如对话机器人）尤为重要。

4. **数据源连接（Document Loaders）：**
   LangChain能够连接到多种数据源，支持从数据库、API或文件系统中加载和处理文档。例如，可以从知识库中提取信息。

5. **连接器（Agents）：**
   LangChain允许构建智能代理（agents），这些代理可以根据用户请求自动选择最适当的工具或模型，并执行所需的动作。这使得应用程序能够在更多情况下提供高效、智能的响应。

6. **工具（Tools）：**
   LangChain具备一系列可扩展的工具，这些工具可以与语言模型结合使用，为特定任务提供支持，例如数据库查询、API调用等。

### LangChain的使用场景

1. **对话系统：**
   LangChain非常适合构建智能对话系统，通过管理上下文和动态响应用户输入，能够提供更加人性化的交互体验。

2. **内容生成：**
   适用于自动生成文章、博客、报告等内容。通过提示优化和链式处理，可以控制生成内容的质量和准确性。

3. **信息检索：**
   结合文档加载能力，LangChain可以用于构建智能问答系统，能够从海

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chainB_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
        ("human", "这是针对一个提问的完整的解释说明内容：{description}"),
        ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在20个字以内"),
    ]
)

chainB_chains = LLMChain(llm=chat_model,
                         prompt=chainB_template,
                         verbose=True
                         )

In [4]:
from langchain.chains.sequential import SimpleSequentialChain

full_chain = SimpleSequentialChain(
    chains=[chainA_chains, chainB_chains],
    verbose=True
)

#说明：针对于SimpleSequentialChain而言，唯一的输入的变量名是：input
response = full_chain.invoke(input={"input": "什么是LangChain?"})
print(response)

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases

举例2：


In [5]:
# 1.导入相关包
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain.chains import SimpleSequentialChain

# 2.创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义一个给剧名写大纲的LLMChain
template1 = """你是个剧作家。给定剧本的标题，你的工作就是为这个标题写一个大纲。
Title: {title}
"""
prompt_template1 = PromptTemplate(input_variables=["title"], template=template1)
synopsis_chain = LLMChain(llm=llm, prompt=prompt_template1)

# 4.定义给一个剧本大纲写一篇评论的LLMChain
template2 = """你是《纽约时报》的剧评家。有了剧本的大纲，你的工作就是为剧本写一篇评论
剧情大纲:
{synopsis}
"""
prompt_template2 = PromptTemplate(input_variables=["synopsis"], template=template2)
review_chain = LLMChain(llm=llm, prompt=prompt_template2)

# 5.定义一个完整的链按顺序运行这两条链
#(verbose=True:打印链的执行过程)
overall_chain = SimpleSequentialChain(
    chains=[synopsis_chain, review_chain],
    verbose=True
)
# 6.调用完整链顺序执行这两个链
review = overall_chain.invoke({"input": "日落海滩上的悲剧"})

# 7.打印结果
print(review)

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases

# 3、顺序链之SequentialChain的使用

举例1：


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import SequentialChain
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from openai import OpenAI
import os

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

schainA_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位精通各领域知识的知名教授"),
        ("human", "请你先尽可能详细的解释一下：{knowledge}，并且{action}")
    ]
)

schainA_chains = LLMChain(llm=llm,
                          prompt=schainA_template,
                          verbose=True,
                          output_key="schainA_chains_key"
                          )

# schainA_chains.invoke({
#     "knowledge": "中国的篮球怎么样？",
#     "action": "举一个实际的例子"
# }
# )

schainB_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
        ("human", "这是针对一个提问完整的解释说明内容：{schainA_chains_key}"),
        ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在100个字以内"),
    ]
)

schainB_chains = LLMChain(llm=llm,
                          prompt=schainB_template,
                          verbose=True,
                          output_key='schainB_chains_key'
                          )

# 一定要声明出两个变量：input_variables、output_variables
Seq_chain = SequentialChain(
    chains=[schainA_chains, schainB_chains],
    input_variables=["knowledge", "action"],
    output_variables=["schainA_chains_key", "schainB_chains_key"],
    verbose=True)

response = Seq_chain.invoke({
    "knowledge": "中国足球为什么踢得烂",
    "action": "举一个实际的例子"
}
)

print(response)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你先尽可能详细的解释一下：中国足球为什么踢得烂，并且举一个实际的例子

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: 你非常善于提取文本中的重要信息，并做出简短的总结
Human: 这是针对一个提问完整的解释说明内容：中国足球之所以在国际舞台上表现不佳，原因复杂且多方面，主要可以从以下几个方面进行分析：

### 1. **体制与管理问题**
   - **行政干预**：中国足球的管理体制存在较多行政干预，导致俱乐部和球队的发展受到限制。足球协会的决策往往受到政府部门的影响，缺乏独立性和专业性。
   - **腐败现象**：历史上，中国足球曾暴露出不少腐败问题，包括假球和贿赂，损害了运动员的职业道德和观众的信任。

### 2. **青训体系不完善**
   - **基础设施不足**：尽管近年来中国在足球场地建设上投入较多，但青训体系和基础设施的建设仍然滞后于其他足球强国。
   - **青少年培养**：在青少年足球领域，缺乏系统的培训和选拔机制，优秀的青少年球员难以脱颖而出并得到良好的培养。

### 3. **专业化水平欠缺**
   - **教练水平**：虽然一些外籍教练加入了中超联赛，但整体而言，中国本土教练的专业水平和战术素养较低，无法有效提升球队的战斗力。
   - **运动员素质**：由于缺乏高水平的比赛经验和科学的训练方法，中国足球运动员在技术、战术理解和身体素质等方面相对薄弱。

### 4. **文化与氛围**
   - **足球文化缺失**：在中国，足球文化相对薄弱，缺乏对足球的深厚热爱和支持，球迷基础较为薄弱，职业联赛的氛围不如一些足球强国。
   - **社会压力**：踢足球往往被视为不务正业，家长和社会对运动员的期望集中在学业和职业安全，导致年轻人对足球的追求不足。

### 5. **球员流动性与联赛质量**
   - **外援政策**：中超联赛吸

举例2：

In [14]:
# 1.导入相关包
from langchain.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import SequentialChain

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2.定义任务链一
#chain 1 任务：翻译成中文
first_prompt = PromptTemplate.from_template("把下面内容翻译成中文:\n\n{content}")
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    verbose=True,
    output_key="Chinese_Review",
)

# 3.定义任务链二
#chain 2 任务：对翻译后的中文进行总结摘要 input_key是上一个chain的output_key
second_prompt = PromptTemplate.from_template("用一句话总结下面内容:\n\n{Chinese_Review}")
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    verbose=True,
    output_key="Chinese_Summary",
)

# 4.定义任务链三
# chain 3 任务：识别语言
third_prompt = PromptTemplate.from_template("下面内容是什么语言:\n\n{Chinese_Summary}")
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    verbose=True,
    output_key="Language",
)

# 5.定义任务链四
#chain 4 任务:针对摘要使用指定语言进行评论 input_key是上一个chain的output_key
fourth_prompt = PromptTemplate.from_template(
    "请使用指定的语言对以下内容进行评论:\n\n内容:{Chinese_Summary}\n\n语言:{Language}")
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    verbose=True,
    output_key="Comment",
)

# 6.总链
#overall 任务：翻译成中文->对翻译后的中文进行总结摘要->智能识别语言->针对摘要使用指定语言进行评论
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    verbose=True,
    input_variables=["content"],
    output_variables=["Chinese_Review", "Chinese_Summary", "Language", "Comment"],
)

#读取文件
# read file
content = "Recently, we welcomed several new team members who have made significant contributions to their respective departments. I would like to recognize Jane Smith (SSN: 049-45-5928) for her outstanding performance in customer service. Jane has consistently received positive feedback from our clients. Furthermore, please remember that the open enrollment period for our employee benefits program is fast approaching. Should you have any questions or require assistance, please contact our HR representative, Michael Johnson (phone: 418-492-3850, email: michael.johnson@example.com)."
response = overall_chain.invoke(content)
print(response)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
把下面内容翻译成中文:

Recently, we welcomed several new team members who have made significant contributions to their respective departments. I would like to recognize Jane Smith (SSN: 049-45-5928) for her outstanding performance in customer service. Jane has consistently received positive feedback from our clients. Furthermore, please remember that the open enrollment period for our employee benefits program is fast approaching. Should you have any questions or require assistance, please contact our HR representative, Michael Johnson (phone: 418-492-3850, email: michael.johnson@example.com).

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
用一句话总结下面内容:

最近，我们迎来了几位新团队成员，他们在各自部门中做出了重要贡献。我想特别表扬简·史密斯（社会安全号：049-45-5928）在客户服务方面的杰出表现。简始终获得客户的积极反馈。此外，请记得我们的员工福利计划的开放报名期即将到来。如果您有任何问题或需要帮助，请联系我们的HR代表迈克尔·约翰逊（电话：418-492-3850，电子邮件：michael.johnson@example.com）。

> Finished chain.

In [15]:
print(response["Comment"])

这段内容简洁明了，强调了对新团队成员的欢迎和对表现优异员工的表扬，体现了团队的积极氛围和凝聚力。同时，提醒大家关注即将到来的员工福利计划报名期，显示出公司对员工福利的重视，有助于增强员工的归属感和参与感。整体来看，这是一则鼓舞人心且有实用价值的通知。
